# Chapter 0 · LLM 归一化：PyTorch 数据实验

本 notebook 与 [本章 README](./README.md) 的第 1–5 节一一对应。读者应先知道张量的形状、最后一维和矩阵乘法；不需要训练模型。运行环境需要 **PyTorch（本例在 2.11.0 验证）** 和 Jupyter。按顺序执行所有单元格。

**学习目标**：观察同一组数据在 LayerNorm、RMSNorm、ScaleNorm 前后的均值/RMS/L2；观察 QK Norm 如何改变注意力分数；理解 DeepNorm 的残差缩放为何不能等同于完整算法。

本实验用固定数据，便于手算与核对。每个代码单元后面有“读输出”的说明。

## 0. 准备数据与统计工具

输入形状为 `[batch=1, seq_len=4, hidden_size=4]`。四行分别是普通向量、整体加 10 的向量、含负数的向量、全零向量。归一化统计量应沿最后一维计算，也就是每个 token 单独算。

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

torch.set_printoptions(precision=3, sci_mode=False)
x = torch.tensor([[[1., 2., 3., 4.],
                   [11., 12., 13., 14.],
                   [-4., -2., 0., 2.],
                   [0., 0., 0., 0.]]])
d = x.shape[-1]

def summarize(name, t):
    print(name)
    print(t.detach())
    print('mean:', t.mean(dim=-1).detach())
    print('RMS: ', t.square().mean(dim=-1).sqrt().detach())
    print('L2:  ', t.norm(p=2, dim=-1).detach())

print('PyTorch:', torch.__version__, '| shape:', tuple(x.shape))
summarize('原始输入', x)

PyTorch: 2.11.0 | shape: (1, 4, 4)
原始输入
tensor([[[ 1.,  2.,  3.,  4.],
         [11., 12., 13., 14.],
         [-4., -2.,  0.,  2.],
         [ 0.,  0.,  0.,  0.]]])
mean: tensor([[ 2.500, 12.500, -1.000,  0.000]])
RMS:  tensor([[ 2.739, 12.550,  2.449,  0.000]])
L2:   tensor([[ 5.477, 25.100,  4.899,  0.000]])


**读输出：**前两行均值分别为 2.5 和 12.5，但元素间距相同。接下来观察“减均值”是否消除整体加 10 的影响。全零行可检查除零保护。`mean(dim=-1)`、`square().mean(dim=-1).sqrt()` 和 `norm(p=2, dim=-1)` 分别得到均值、均方根和 L2 长度。

## 1. LayerNorm：减均值，再按标准差缩放

对应 README 的 **1. LayerNorm**。`nn.LayerNorm(normalized_shape=d, eps=1e-5)` 在最后 `d` 维中归一化；这里 `d` 是整数，因此只处理最后一维。默认 `elementwise_affine=True, bias=True`，会创建形状为 `[d]` 的 `weight` 和 `bias`，初值分别为 1 和 0。`eps` 加在方差上，防止除零。

In [2]:
ln = nn.LayerNorm(normalized_shape=d, eps=1e-5)
y_ln = ln(x)
summarize('LayerNorm 输出', y_ln)
print('可学习参数:', [(name, tuple(p.shape)) for name, p in ln.named_parameters()])
print('前两行相同:', torch.allclose(y_ln[0, 0], y_ln[0, 1], atol=1e-5))

LayerNorm 输出
tensor([[[-1.342, -0.447,  0.447,  1.342],
         [-1.342, -0.447,  0.447,  1.342],
         [-1.342, -0.447,  0.447,  1.342],
         [ 0.000,  0.000,  0.000,  0.000]]])
mean: tensor([[0., 0., 0., 0.]])
RMS:  tensor([[1.000, 1.000, 1.000, 0.000]])
L2:   tensor([[2.000, 2.000, 2.000, 0.000]])
可学习参数: [('weight', (4,)), ('bias', (4,))]
前两行相同: True


**读输出：**前两行变成同一向量，说明 LayerNorm 消除了整体平移。非零行的均值约为 0，RMS 约为 1。全零行仍是零；学习后 `weight/bias` 可以改变输出的均值和尺度。不要把“初始化状态的输出统计量”误认为训练后永远成立。

## 2. RMSNorm：保留均值，只控制均方根

对应 README 的 **2. RMSNorm**。`nn.RMSNorm(normalized_shape=d, eps=1e-5)` 同样只处理最后一维。默认 `elementwise_affine=True`，只创建 `[d]` 的 `weight`，没有 `bias`。若不显式传 `eps`，PyTorch 的默认值是 `None`，会按计算 dtype 选择 machine epsilon；这里固定 `1e-5` 以方便对照。

In [3]:
rms = nn.RMSNorm(normalized_shape=d, eps=1e-5)
y_rms = rms(x)
summarize('RMSNorm 输出', y_rms)
print('可学习参数:', [(name, tuple(p.shape)) for name, p in rms.named_parameters()])
print('前两行相同:', torch.allclose(y_rms[0, 0], y_rms[0, 1], atol=1e-5))

RMSNorm 输出
tensor([[[ 0.365,  0.730,  1.095,  1.461],
         [ 0.877,  0.956,  1.036,  1.116],
         [-1.633, -0.816,  0.000,  0.816],
         [ 0.000,  0.000,  0.000,  0.000]]])
mean: tensor([[ 0.913,  0.996, -0.408,  0.000]])
RMS:  tensor([[1.000, 1.000, 1.000, 0.000]])
L2:   tensor([[2.000, 2.000, 2.000, 0.000]])
可学习参数: [('weight', (4,))]
前两行相同: False


**读输出：**非零行的 RMS 约为 1，均值却不为 0；整体加 10 后输出也不同。这就是 RMSNorm 与 LayerNorm 最直观的差别。全零行保持有限值。

## 3. ScaleNorm：控制 L2 长度

对应 README 的 **3. ScaleNorm**。PyTorch 没有同名 `nn.ScaleNorm` 标准模块；可以用 `F.normalize(x, p=2, dim=-1, eps=1e-6)` 做 L2 归一化，再乘标量 $g$。**务必写 `dim=-1`**：`F.normalize` 的默认 `dim=1` 会在本例中沿 `seq_len`，不是隐藏维度。`eps` 通过 `max(L2, eps)` 保护零向量。

In [4]:
g = 1.0  # 为观察 L2=1，暂固定；训练时可使用 nn.Parameter。
y_scale = g * F.normalize(x, p=2, dim=-1, eps=1e-6)
summarize('ScaleNorm 输出（g=1）', y_scale)
print('非零行 L2:', y_scale.norm(p=2, dim=-1)[0, :3])
# 可学习写法：self.g = nn.Parameter(torch.tensor(1.0))，放在 nn.Module 内。

ScaleNorm 输出（g=1）
tensor([[[ 0.183,  0.365,  0.548,  0.730],
         [ 0.438,  0.478,  0.518,  0.558],
         [-0.816, -0.408,  0.000,  0.408],
         [ 0.000,  0.000,  0.000,  0.000]]])
mean: tensor([[ 0.456,  0.498, -0.204,  0.000]])
RMS:  tensor([[0.500, 0.500, 0.500, 0.000]])
L2:   tensor([[1.000, 1.000, 1.000, 0.000]])
非零行 L2: tensor([1.000, 1.000, 1.000])


**读输出：**前三行的 L2 长度约为 1，均值不一定为 0；全零行的 L2 仍为 0。与 RMSNorm 比较：对维度数为 `d` 的非零向量，`RMS = L2 / sqrt(d)`，所以初始化权重为 1 的 RMSNorm 输出 L2 约为 `sqrt(d)`，而这里 ScaleNorm 的输出 L2 约为 `g=1`。

### API 小实验：`dim` 写错会怎样？

下列单元格演示默认 `dim=1` 的危险。先比较每个 token 的 L2；正确设置应让前三个 token 的 L2 接近 1。

In [5]:
wrong = F.normalize(x, p=2)  # 默认 dim=1：沿 token/seq 维度
right = F.normalize(x, p=2, dim=-1)
print('省略 dim 时各 token 的 L2:', wrong.norm(p=2, dim=-1))
print('dim=-1 时各 token 的 L2:', right.norm(p=2, dim=-1))

省略 dim 时各 token 的 L2: tensor([[0.398, 1.919, 0.401, 0.000]])
dim=-1 时各 token 的 L2: tensor([[1.000, 1.000, 1.000, 0.000]])


## 4. QK Norm：注意力内部的归一化

对应 README 的 **4. QK Norm**。这里演示原始论文思路的 **L2 版**，并非所有模型的 QK Norm 实现。Q/K 形状为 `[batch, heads, seq_len, head_dim]`；`F.normalize(..., dim=-1)` 处理每个头的向量。先计算原始 `q @ k.transpose(-2, -1)`，再观察归一化后的 logits 和 softmax。`transpose(-2, -1)` 只交换 key 的 `seq_len` 与 `head_dim`，保留 batch/head 维度：`[B,H,S_k,D] → [B,H,D,S_k]`；因此分数形状为 `[B,H,S_q,S_k]`。softmax 要沿最后一维 `S_k` 计算，即每个 query 对所有 key 的概率。

In [6]:
q = torch.tensor([[[[10., 0.], [0., 1.]]]])
k = torch.tensor([[[[10., 0.], [0., 1.]]]])
raw_scores = q @ k.transpose(-2, -1)
q_unit = F.normalize(q, p=2, dim=-1, eps=1e-6)
k_unit = F.normalize(k, p=2, dim=-1, eps=1e-6)
norm_scores = q_unit @ k_unit.transpose(-2, -1)
print('原始 logits:', raw_scores)
print('归一化后 logits（g=1）:', norm_scores)
print('原始 softmax:', raw_scores.softmax(dim=-1))
print('归一化后 softmax:', norm_scores.softmax(dim=-1))

原始 logits: tensor([[[[100.,   0.],
          [  0.,   1.]]]])
归一化后 logits（g=1）: tensor([[[[1., 0.],
          [0., 1.]]]])
原始 softmax: tensor([[[[1.000, 0.000],
          [0.269, 0.731]]]])
归一化后 softmax: tensor([[[[0.731, 0.269],
          [0.269, 0.731]]]])


**读输出：**原始最大 logit 是 100，第一行 softmax 几乎变成 `[1,0]`；L2 归一化后最大 logit 是 1，概率约为 `[0.731,0.269]`。这只是刻意选的示例，不表示 QK Norm 在真实模型中总会得到这个概率。可学习尺度 $g$ 会再次调整 logit 幅度。

### PyTorch 注意力调用

`F.scaled_dot_product_attention(q,k,v,is_causal=False,scale=1.0)` 返回加权后的 value，**不直接返回 logits/概率**。`scale` 只能是 Python 数值，不能直接传 `nn.Parameter`。若 $g$ 可学习，可以先做 `q_unit * g`，再设 `scale=1.0`。不传 `scale` 时，函数会默认使用 $1/\sqrt{d_k}$。本例无因果掩码；自回归模型应设置 `is_causal=True`。

In [7]:
v = torch.tensor([[[[1., 0.], [0., 1.]]]])
g_qk = nn.Parameter(torch.tensor(1.0))
attention_out = F.scaled_dot_product_attention(
    q_unit * g_qk, k_unit, v, is_causal=False, scale=1.0
)
print('注意力输出:', attention_out)
print('与手算概率一致:', torch.allclose(attention_out, norm_scores.softmax(dim=-1) @ v))

注意力输出: tensor([[[[0.731, 0.269],
          [0.269, 0.731]]]],
       grad_fn=<ScaledDotProductFlashAttentionForCpuBackward0>)
与手算概率一致: True


## 5. DeepNorm：残差路径与初始化配合

对应 README 的 **5. DeepNorm**。这里仅观察 `LayerNorm(alpha*x + branch)` 中 `alpha` 对数据的影响。`alpha=1.5` 是**演示值，不是论文系数**。完整 DeepNorm 必须按模型深度和编码器/解码器结构选择系数，并使用配套的权重初始化；PyTorch 没有独立 `nn.DeepNorm` 标准模块。

In [8]:
residual_x = torch.tensor([[[1., 2., 3., 4.]]])
branch = torch.tensor([[[2., -1., 1., -2.]]])
post_norm = nn.LayerNorm(4, eps=1e-5)
plain_input = residual_x + branch
alpha = 1.5
scaled_input = alpha * residual_x + branch
summarize('普通残差和，alpha=1', plain_input)
summarize('缩放后的残差和，alpha=1.5', scaled_input)
summarize('LayerNorm(alpha*x + branch)', post_norm(scaled_input))

普通残差和，alpha=1
tensor([[[3., 1., 4., 2.]]])
mean: tensor([[2.500]])
RMS:  tensor([[2.739]])
L2:   tensor([[5.477]])
缩放后的残差和，alpha=1.5
tensor([[[3.500, 2.000, 5.500, 4.000]]])
mean: tensor([[3.750]])
RMS:  tensor([[3.953]])
L2:   tensor([[7.906]])
LayerNorm(alpha*x + branch)
tensor([[[-0.200, -1.400,  1.400,  0.200]]])
mean: tensor([[0.000]])
RMS:  tensor([[1.000]])
L2:   tensor([[2.000]])


**读输出：**改变残差路径权重后，送入 LayerNorm 的向量改变，归一化后的方向也改变。由于 LayerNorm 会重新调整尺度，只比较最终输出的 RMS 并不足以理解 DeepNorm；训练稳定性还依赖深度相关系数和初始化。

## 6. 对照练习

1. 把第 0 节的第二行从 `[11,12,13,14]` 改为 `[101,102,103,104]`，预测三种隐藏状态归一化的哪一种仍与第一行输出相同。
2. 把第 4 节的可学习尺度 `g_qk` 从 1 改为 5，预测 softmax 会更尖锐还是更平缓。
3. 将第 3 节 `g` 设为 2，预测 ScaleNorm 输出的 L2 长度。

先预测，再重跑相关单元格。参考答案见下一节。

### 练习答案与常见坑

1. LayerNorm 仍与第一行相同，因为减均值消除了整体平移；RMSNorm 和 ScaleNorm 一般不同。
2. 分数差距放大，softmax 通常更尖锐。`scale` 要保持 `1.0`，避免额外缩放。
3. 非零输入的 L2 长度约为 2；全零行仍为 0。

常见坑：把 `F.normalize` 的默认 `dim=1` 当成隐藏维度；把 `F.scaled_dot_product_attention` 的输出当成概率矩阵；把 DeepNorm 的残差示例误认为完整论文算法。

**可选延伸：**尝试 `nn.RMSNorm(head_dim)` 处理逐头 Q/K，并记录它与 L2 版 QK Norm 的输出差别；模型使用 RoPE 时还需核对归一化与 RoPE 的顺序。

## 7. API 参数速查与下一步

| API | 关键参数 | 本实验的取值或说明 |
| --- | --- | --- |
| `nn.LayerNorm` | `normalized_shape`, `eps`, `elementwise_affine`, `bias` | `d`, `1e-5`, `True`, `True`；默认有 `weight/bias` |
| `nn.RMSNorm` | `normalized_shape`, `eps`, `elementwise_affine` | `d`, `1e-5`, `True`；默认仅有 `weight` |
| `F.normalize` | `p`, `dim`, `eps` | `2`, `-1`, `1e-6`；不创建可学习参数 |
| `Tensor.norm` | `p`, `dim`, `keepdim` | 常用 `2`, `-1`, `True`；`keepdim=True` 便于广播 |
| `F.scaled_dot_product_attention` | `is_causal`, `scale`, `dropout_p` | 本例 `False`, `1.0`, `0.0`；自回归通常用 `is_causal=True` |

下一步可回到 [README 的公式与论文链接](./README.md) 核对各方法的定义。若要比较性能，应控制设备、dtype、输入形状与内核；本 notebook 只做概念实验。